# Tutorial 1: Train an AART Model (SomaScan → Olink)

This notebook walks through training AART on paired CKB proteomics data to impute Olink protein levels from SomaScan measurements.

**AART** (Anchor-Aware Residual Translator) combines:
1. **Anchor branch** — per-protein Ridge regression using biologically matched SomaScan aptamers
2. **Residual branch** — global PCA + Ridge model on the full SomaScan feature space
3. **Reliability gate** — closed-form per-protein weighting that balances anchor vs. residual

**Data**: CKB cohort (n ≈ 3,975 paired samples), 2,168 Olink proteins, 2,731 SomaScan aptamers.

In [14]:
import sys
from pathlib import Path

TUTORIAL_DIR = Path(".").resolve()
PROJECT_ROOT = TUTORIAL_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.models.aart.anchors import (
    AnchorBundle,
    ResidualBundle,
    apply_gate,
    compute_residual_prior,
    fit_closed_form_gate,
    fit_residual_bundle,
)
from src.models.aart.mapping import build_candidate_table, canonicalize_symbol, seqid_to_probe_id
from src.models.aart.preprocessing import (
    PreprocessingBundle,
    fit_preprocessing,
    transform_soma,
    inverse_transform_targets,
    dataframe_ids,
    dataframe_targets,
)
from src.models.aart.metrics import median_gene_pearson

print(f"Project root: {PROJECT_ROOT}")

Project root: /nfs/roberts/pi/pi_sz622/yc2352/AART/github_ready


## 1. Load paired proteomics data

In [15]:
DATA_DIR = PROJECT_ROOT / "data" / "CKB"

olink_df = pd.read_csv(DATA_DIR / "olink_overlap.csv")
soma_df = pd.read_csv(DATA_DIR / "somascan_overlap.csv")

print(f"Olink:    {olink_df.shape[0]} samples x {olink_df.shape[1] - 1} proteins")
print(f"SomaScan: {soma_df.shape[0]} samples x {soma_df.shape[1] - 1} aptamers")
print(f"ID column: '{olink_df.columns[0]}'")
print(f"\nOlink proteins (first 5):    {list(olink_df.columns[1:6])}")
print(f"SomaScan aptamers (first 5): {list(soma_df.columns[1:6])}")

Olink:    3976 samples x 2168 proteins
SomaScan: 3976 samples x 2731 aptamers
ID column: 'csid'

Olink proteins (first 5):    ['NPPB', 'TNNI3', 'HNRNPK', 'CEBPB', 'VIM']
SomaScan aptamers (first 5): ['seq.10000.28', 'seq.10010.10', 'seq.10046.55', 'seq.10047.12', 'seq.10049.112']


## 2. Build the aptamer-protein annotation table

The mapping between SomaScan aptamers and Olink proteins is derived from
[Wang et al. 2025 (Nature Communications)](https://doi.org/10.1038/s41467-025-56935-2),
Supplementary Data 1. Each row links a UniProt ID to an Olink gene symbol
and a SomaScan assay ID.

AART's `build_candidate_table` expects an annotation with columns
`probe_id`, `symbol_values`, `alias_values`, and `genename_values`.
We convert the Excel file to this format.

In [16]:
mapping_raw = pd.read_excel(
    DATA_DIR / "41467_2025_56935_MOESM4_ESM.xlsx",
    sheet_name="Supplementary_Data_1",
)
print(f"Mapping table: {mapping_raw.shape[0]} rows")
print(f"Columns: {list(mapping_raw.columns)}")
mapping_raw.head()

Mapping table: 2749 rows
Columns: ['uniprot_id', 'olink_id', 'somascan_id', 'rho_olink_soma_ANML', 'rho_olink_soma_non_ANML', 'r_olink_soma_ANML', 'r_olink_soma_non_ANML']


,uniprot_id,olink_id,somascan_id,rho_olink_soma_ANML,rho_olink_soma_non_ANML,r_olink_soma_ANML,r_olink_soma_non_ANML
0,A1KZ92,PXDNL,seq.11324.3,0.011688,0.045635,0.015156,0.024351
1,A6NCE7,MAP1LC3B2,seq.20965.18,0.002644,-0.008807,-0.009787,-0.009422
2,A6NGN9,IGLON5,seq.6478.2,0.009516,0.008595,-0.017359,-0.006946
3,A6NHS7,MANSC4,seq.9578.263,0.686147,0.658314,0.644446,0.626073
4,A6NI73,LILRA5,seq.8766.29,0.687786,0.856891,0.726757,0.862059


In [17]:
# Convert somascan_id (e.g. "seq.11324.3") to probe_id (e.g. "11324-3")
# Group by probe_id and aggregate all mapped olink symbols (pipe-separated)
mapping_raw["probe_id"] = mapping_raw["somascan_id"].apply(seqid_to_probe_id)

annotation = (
    mapping_raw
    .groupby("probe_id", sort=False)
    .agg(
        symbol_values=("olink_id", lambda s: "|".join(s.dropna().unique())),
        alias_values=("olink_id", lambda s: "|".join(s.dropna().unique())),
        genename_values=("uniprot_id", lambda s: "|".join(s.dropna().unique())),
    )
    .reset_index()
)

n_unique_probes = annotation.shape[0]
n_unique_olink = mapping_raw["olink_id"].nunique()
print(f"Annotation: {n_unique_probes} SomaScan probes mapped to {n_unique_olink} unique Olink symbols")
annotation.head()

Annotation: 2731 SomaScan probes mapped to 2168 unique Olink symbols


,probe_id,symbol_values,alias_values,genename_values
0,11324-3,PXDNL,PXDNL,A1KZ92
1,20965-18,MAP1LC3B2,MAP1LC3B2,A6NCE7
2,6478-2,IGLON5,IGLON5,A6NGN9
3,9578-263,MANSC4,MANSC4,A6NHS7
4,8766-29,LILRA5,LILRA5,A6NI73


## 3. Train/test split

We hold out 20% of samples for evaluation.

In [18]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.1  # fraction of training set used for gate fitting

ids = olink_df["csid"].values
train_idx, test_idx = train_test_split(
    np.arange(len(ids)), test_size=TEST_SIZE, random_state=RANDOM_STATE
)
train_idx.sort()
test_idx.sort()

olink_train = olink_df.iloc[train_idx].reset_index(drop=True)
olink_test = olink_df.iloc[test_idx].reset_index(drop=True)
soma_train = soma_df.iloc[train_idx].reset_index(drop=True)
soma_test = soma_df.iloc[test_idx].reset_index(drop=True)

print(f"Train: {len(train_idx)} samples")
print(f"Test:  {len(test_idx)} samples")

Train: 3180 samples
Test:  796 samples


## 4. Preprocessing

Split training data into train/val **before** fitting scalers to avoid leaking
validation statistics into the standardization. SomaScan values are log-transformed
and standardized; Olink values are standardized directly.

In [19]:
# Split training into train/val FIRST, then fit scalers on train-only
from src.models.aart.preprocessing import transform_olink

n_full = len(olink_train)
val_count = int(n_full * VAL_SIZE)
perm = np.random.RandomState(RANDOM_STATE).permutation(n_full)
val_indices = np.sort(perm[:val_count])
train_indices = np.sort(perm[val_count:])

olink_tr = olink_train.iloc[train_indices].reset_index(drop=True)
olink_va = olink_train.iloc[val_indices].reset_index(drop=True)
soma_tr = soma_train.iloc[train_indices].reset_index(drop=True)
soma_va = soma_train.iloc[val_indices].reset_index(drop=True)

# Fit scalers on train-only (no validation data touches the scaler)
preprocessing, x_train, y_train = fit_preprocessing(olink_tr, soma_tr)

# Transform val and test using train-fitted scalers
x_val = transform_soma(soma_va, preprocessing)
y_val = transform_olink(olink_va, preprocessing)
x_test = transform_soma(soma_test, preprocessing)
y_test_raw = dataframe_targets(olink_test)
ids_test = dataframe_ids(olink_test)

gene_names = preprocessing.gene_names
soma_feature_names = preprocessing.soma_feature_names

print(f"x_train: {x_train.shape}  y_train: {y_train.shape}")
print(f"x_val:   {x_val.shape}    y_val:   {y_val.shape}")
print(f"x_test:  {x_test.shape}")
print(f"\n{len(gene_names)} Olink targets, {len(soma_feature_names)} SomaScan features")

x_train: (2862, 2731)  y_train: (2862, 2168)
x_val:   (318, 2731)    y_val:   (318, 2168)
x_test:  (796, 2731)

2168 Olink targets, 2731 SomaScan features


## 5. Build candidate mapping table

This links each Olink target gene to its matching SomaScan aptamer(s) via the annotation.

In [20]:
candidates = build_candidate_table(gene_names, soma_feature_names, annotation)

n_genes_with_anchor = candidates["gene"].nunique()
n_one_to_one = candidates.groupby("gene").size().eq(1).sum()
n_multi = n_genes_with_anchor - n_one_to_one

print(f"Candidate links: {len(candidates)} (gene-aptamer pairs)")
print(f"Genes with anchor: {n_genes_with_anchor} / {len(gene_names)}")
print(f"  1-to-1 mappings: {n_one_to_one}")
print(f"  multi mappings:  {n_multi}")
print(f"  no anchor:       {len(gene_names) - n_genes_with_anchor}")
candidates.head(8)

Candidate links: 2749 (gene-aptamer pairs)
Genes with anchor: 2168 / 2168
  1-to-1 mappings: 1696
  multi mappings:  472
  no anchor:       0


,gene,protein_symbol,gene_idx,apt_col,apt_idx,probe_id,match_type,symbol_values,alias_values,genename_values
0,NPPB,NPPB,0,seq.16751.15,665,16751-15,direct,NPPB,NPPB,P16860
1,NPPB,NPPB,0,seq.3723.1,1725,3723-1,direct,NPPB,NPPB,P16860
2,NPPB,NPPB,0,seq.7655.11,2333,7655-11,direct,NPPB,NPPB,P16860
3,TNNI3,TNNI3,1,seq.5441.67,2015,5441-67,direct,TNNI3,TNNI3,P19429
4,TNNI3,TNNI3,1,seq.5930.54,2129,5930-54,direct,TNNI3,TNNI3,P19429
5,HNRNPK,HNRNPK,2,seq.19333.4,899,19333-4,direct,HNRNPK,HNRNPK,P61978
6,CEBPB,CEBPB,3,seq.15675.3,620,15675-3,direct,CEBPB,CEBPB,P17676
7,CEBPB,CEBPB,3,seq.22953.85,1205,22953-85,direct,CEBPB,CEBPB,P17676


## 6. Fit AART components

### 6a. Anchor branch
Per-protein Ridge regression on matched SomaScan aptamers.

In [21]:
ANCHOR_ALPHA_SINGLE = 0.01
ANCHOR_ALPHA_MULTI = 0.1

anchor_bundle = AnchorBundle.fit(
    x_train=x_train,
    y_train=y_train,
    gene_names=gene_names,
    candidates=candidates,
    alpha_single=ANCHOR_ALPHA_SINGLE,
    alpha_multi=ANCHOR_ALPHA_MULTI,
)

anchor_train = anchor_bundle.predict(x_train)
anchor_val = anchor_bundle.predict(x_val)
anchor_test = anchor_bundle.predict(x_test)

anchor_meta = anchor_bundle.metadata_frame()
n_anchored = int(anchor_meta["has_anchor"].sum())
median_anchor_corr = anchor_meta.loc[anchor_meta["has_anchor"] == 1, "train_anchor_corr"].median()

print(f"Anchor models fitted: {n_anchored} / {len(gene_names)} proteins")
print(f"Median training anchor correlation: {median_anchor_corr:.3f}")

Anchor models fitted: 2168 / 2168 proteins
Median training anchor correlation: 0.271


### 6b. Residual branch
PCA + Ridge on the full SomaScan space, trained on the residuals after removing the anchor prediction.

In [22]:
N_COMPONENTS = 256
RESIDUAL_ALPHA = 100.0

residual_train = y_train - anchor_train

residual_bundle = fit_residual_bundle(
    x_train=x_train,
    residual_train=residual_train,
    n_components=N_COMPONENTS,
    alpha=RESIDUAL_ALPHA,
    random_state=RANDOM_STATE,
)

residual_val_pred = residual_bundle.predict(x_val)
residual_test_pred = residual_bundle.predict(x_test)

print(f"PCA components: {residual_bundle.pca.n_components_}")
print(f"Variance explained: {residual_bundle.pca.explained_variance_ratio_.sum():.2%}")

PCA components: 256
Variance explained: 67.61%


### 6c. Reliability gate
Closed-form per-protein gate weight that blends anchor and residual predictions. The prior trusts the residual more when the anchor is weak.

In [23]:
GATE_LAMBDA = 1.0

residual_prior = compute_residual_prior(anchor_bundle)

gate = fit_closed_form_gate(
    residual_val_pred=residual_val_pred,
    anchor_val=anchor_val,
    y_val=y_val,
    residual_prior=residual_prior,
    lam=GATE_LAMBDA,
)

print(f"Gate weights — mean: {gate.mean():.3f}, median: {np.median(gate):.3f}")
print(f"  min: {gate.min():.3f}, max: {gate.max():.3f}")
print(f"  gate=0 (trust anchor fully): {(gate == 0).sum()} proteins")
print(f"  gate=1 (trust residual fully): {(gate == 1).sum()} proteins")

Gate weights — mean: 0.742, median: 0.858
  min: 0.000, max: 1.000
  gate=0 (trust anchor fully): 66 proteins
  gate=1 (trust residual fully): 418 proteins


## 7. Predict on test set

Combine anchor + gate × residual, then inverse-transform back to the original Olink scale.

In [24]:
# AART prediction: anchor + gate * residual
aart_test_std = apply_gate(anchor_test, residual_test_pred, gate)
aart_test_raw = inverse_transform_targets(aart_test_std, preprocessing)

# Direct 1-to-1 baseline: anchor only (no residual, no gate)
direct_test_std = anchor_test
direct_test_raw = inverse_transform_targets(direct_test_std, preprocessing)

# Quick check
aart_score = median_gene_pearson(y_test_raw, aart_test_raw)
direct_score = median_gene_pearson(y_test_raw, direct_test_raw)
print(f"Median per-protein Pearson r on test set:")
print(f"  AART:           {aart_score:.4f}")
print(f"  Direct 1-to-1:  {direct_score:.4f}")

Median per-protein Pearson r on test set:
  AART:           0.6840
  Direct 1-to-1:  0.2765


## 8. Save artifacts

Save the trained model components and predictions for use in Notebook 2.

In [25]:
OUTPUT_DIR = TUTORIAL_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Save model artifacts
with open(OUTPUT_DIR / "preprocessing.pkl", "wb") as f:
    pickle.dump(preprocessing, f)
with open(OUTPUT_DIR / "anchor_bundle.pkl", "wb") as f:
    pickle.dump(anchor_bundle, f)
with open(OUTPUT_DIR / "residual_bundle.pkl", "wb") as f:
    pickle.dump(residual_bundle, f)
np.save(OUTPUT_DIR / "gate.npy", gate)

# Save predictions and ground truth
pred_df = pd.DataFrame(aart_test_raw, columns=gene_names)
pred_df.insert(0, "csid", ids_test)
pred_df.to_csv(OUTPUT_DIR / "aart_predictions_test.csv", index=False)

direct_pred_df = pd.DataFrame(direct_test_raw, columns=gene_names)
direct_pred_df.insert(0, "csid", ids_test)
direct_pred_df.to_csv(OUTPUT_DIR / "direct_predictions_test.csv", index=False)

truth_df = pd.DataFrame(y_test_raw, columns=gene_names)
truth_df.insert(0, "csid", ids_test)
truth_df.to_csv(OUTPUT_DIR / "olink_truth_test.csv", index=False)

# Save test indices and mapping for evaluation notebook
np.save(OUTPUT_DIR / "test_idx.npy", test_idx)
annotation.to_csv(OUTPUT_DIR / "annotation.csv", index=False)
candidates.to_csv(OUTPUT_DIR / "candidates.csv", index=False)
anchor_meta.to_csv(OUTPUT_DIR / "anchor_metadata.csv", index=False)

print(f"Artifacts saved to {OUTPUT_DIR}")
print(f"Files: {[f.name for f in sorted(OUTPUT_DIR.iterdir())]}")

Artifacts saved to /nfs/roberts/pi/pi_sz622/yc2352/AART/github_ready/tutorial/outputs
Files: ['aart_predictions_test.csv', 'anchor_bundle.pkl', 'anchor_metadata.csv', 'annotation.csv', 'candidates.csv', 'direct_predictions_test.csv', 'gate.npy', 'histogram_per_sample.png', 'olink_truth_test.csv', 'per_protein_comparison.csv', 'preprocessing.pkl', 'residual_bundle.pkl', 'scatter_aart_vs_direct.png', 'test_idx.npy', 'violin_aart_vs_direct.png', 'violin_stratified_by_concordance.png']
